# CardioVision AI — Protótipo de Classificação de ECG

Interface interativa no notebook para simular o uso clínico do classificador.

Carrega os modelos treinados (`cnn.keras` e `resnet50.keras`) e permite classificar imagens do conjunto de teste.

In [2]:
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from IPython.display import display, Markdown
from tensorflow.keras.applications.resnet50 import preprocess_input

PROCESSED_PATH = Path("../data/processed")
MODELS_PATH = Path("../trained_models")
IMG_SIZE = (224, 224)

assert (PROCESSED_PATH / "metadata.json").exists(), "Execute 01_preprocessing.ipynb primeiro."

import json
class_names = json.loads((PROCESSED_PATH / "metadata.json").read_text(encoding="utf-8"))["class_names"]


@tf.keras.utils.register_keras_serializable(package="CardioVision")
class ResNet50Preprocess(tf.keras.layers.Layer):
    def call(self, inputs):
        return preprocess_input(inputs)


RESNET_CUSTOM_OBJECTS = {
    "preprocess_input": preprocess_input,
    "ResNet50Preprocess": ResNet50Preprocess,
}


def load_trained_model(path: Path, needs_resnet_objects: bool = False):
    if needs_resnet_objects:
        return tf.keras.models.load_model(
            path,
            custom_objects=RESNET_CUSTOM_OBJECTS,
            safe_mode=False,
        )
    return tf.keras.models.load_model(path)


available_models = {}
for name, filename, resnet in [
    ("CNN do zero", "cnn.keras", False),
    ("ResNet50", "resnet50.keras", True),
]:
    path = MODELS_PATH / filename
    if path.exists():
        try:
            available_models[name] = load_trained_model(path, needs_resnet_objects=resnet)
            print(f"✓ {name} carregado")
        except Exception as exc:
            print(f"✗ {name} erro ao carregar: {exc}")
    else:
        print(f"✗ {name} não encontrado — execute o notebook de treino correspondente")

if not available_models:
    raise RuntimeError("Nenhum modelo disponível. Execute os notebooks 02 e/ou 03 primeiro.")

sample_paths = sorted((PROCESSED_PATH / "test").rglob("*.jpg"))
print(f"\n{len(sample_paths)} imagens de teste disponíveis")

✓ CNN do zero carregado

✓ ResNet50 carregado

140 imagens de teste disponíveis


In [ ]:
def run_inference(model, image_path: str, model_name: str):
    img = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)
    batch = tf.expand_dims(arr, axis=0)
    probs = model.predict(batch, verbose=0)[0]
    pred_idx = int(np.argmax(probs))

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].imshow(arr.astype("uint8"))
    axes[0].set_title(Path(image_path).name, fontsize=10)
    axes[0].axis("off")

    colors = ["#2ecc71" if i == pred_idx else "#3498db" for i in range(len(class_names))]
    axes[1].barh([c.replace("_ecg_images", "") for c in class_names], probs, color=colors)
    axes[1].set_xlim(0, 1)
    axes[1].set_xlabel("Probabilidade")
    pred_label = class_names[pred_idx].replace("_ecg_images", "")
    axes[1].set_title(f"{model_name}: {pred_label} ({probs[pred_idx]:.1%})")
    plt.tight_layout()
    plt.show()

    return pred_label, float(probs[pred_idx])


model_dropdown = widgets.Dropdown(
    options=list(available_models.keys()),
    description="Modelo:",
)
image_dropdown = widgets.Dropdown(
    options=[(p.name, str(p)) for p in sample_paths],
    description="ECG:",
    layout=widgets.Layout(width="70%"),
)
classify_btn = widgets.Button(description="Analisar ECG", button_style="success", icon="heart")
result_html = widgets.HTML()
output = widgets.Output()


def on_classify(_):
    with output:
        output.clear_output(wait=True)
        model = available_models[model_dropdown.value]
        label, conf = run_inference(model, image_dropdown.value, model_dropdown.value)
        result_html.value = (
            f"<div style='padding:10px;background:#ecf9f1;border-radius:8px;'>"
            f"<b>Resultado:</b> {label} &nbsp;|&nbsp; <b>Confiança:</b> {conf:.1%}"
            f"</div>"
        )


classify_btn.on_click(on_classify)

display(Markdown("## 🫀 CardioVision — Simulador de Classificação de ECG"))
display(widgets.VBox([model_dropdown, image_dropdown, classify_btn, result_html, output]))

## 🫀 CardioVision — Simulador de Classificação de ECG